# Xavier mini-expert training (generated, self-contained)

Generated by `scripts/training/train_expert.py --backend notebook`. Do not edit the core cell.

**Human steps (free Colab, T4 GPU):**
1. Runtime > Change runtime type > T4 GPU.
2. Open the Files panel (left sidebar) and upload `bundle.zip` to `/content/bundle.zip`
   (drag it onto the panel; nothing is mounted, no Drive access).
3. Runtime > Run all.
4. When finished, download `/content/out/model-q4_k_m.gguf` and `/content/out/train_report.json`
   from the Files panel (right-click > Download).
5. Back on the node: `POST /v1/training/jobs/{id}/artifacts` with both files, or run
   `ollama create` manually, then `scripts/eval/expert_eval.py`.

Base model: `@@BASE_MODEL@@` - expected bundle hash: `@@BUNDLE_HASH@@`


In [ ]:
# 1. Install dependencies and build llama.cpp tools (a few minutes)
!pip install -q @@PIP_PACKAGES@@
!git clone --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q -r /content/llama.cpp/requirements/requirements-convert_hf_to_gguf.txt
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF -DLLAMA_CURL=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j 4 > /dev/null


In [ ]:
# 2. Core (inlined verbatim from scripts/training/expert_core.py)
@@EXPERT_CORE@@


In [ ]:
# 3. Run: validate bundle -> LoRA -> merge -> GGUF f16 -> Q4_K_M -> train_report.json
import json, os
os.environ["LLAMA_CPP_DIR"] = "/content/llama.cpp"
CONFIG = json.loads(r'''@@CONFIG_JSON@@''')

bundle_dir = unzip_bundle("/content/bundle.zip", "/content/bundle")
if bundle_hash(bundle_dir) != CONFIG["expected_bundle_hash"]:
    raise ExpertError("bundle.zip does not match the bundle this notebook was generated for")
report = run_pipeline(bundle_dir, CONFIG["base_model"], "/content/out",
                      CONFIG["hyperparams"], backend="notebook")
print(json.dumps(report, indent=2))
print("\nDONE. Download from the Files panel:")
print("  /content/out/model-q4_k_m.gguf")
print("  /content/out/train_report.json")
